# Lab 7 (Part B): Explainable AI, Adversarial AI and ResNet Classification
**IC-709D | AI Lab** | PyTorch, ResNet-18, Grad-CAM, FGSM

**Aim:** Use a pretrained ResNet-18 to classify images, explain its predictions with Grad-CAM,
attack it with the Fast Gradient Sign Method (FGSM), and analyse (a) how different explanation
techniques compare and (b) how robust the model is against adversarial attacks.

> **Run this notebook on Google Colab** (Runtime > Change runtime type > GPU). It downloads the pretrained
> weights and the CIFAR-10 test images automatically, so internet access is required.

| Section | Topic |
|---|---|
| 1 | ResNet-based image classification (transfer learning basics) |
| 2 | CNN visualisation using Grad-CAM |
| 3 | Adversarial attack using FGSM |
| 4 | Analysis: comparing Explainable AI techniques |
| 5 | Analysis: robustness against adversarial attacks (FGSM vs PGD) |
| 6 | Summary and learning outcomes |

In [ ]:
import os
import time
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import datasets, models, transforms

warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)   # figures are saved here
os.makedirs("data", exist_ok=True)      # downloaded images (do not upload this folder to GitHub)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| Device:", DEVICE)

## 1. ResNet-based image classification
**ResNet-18** is a convolutional neural network with 18 layers. Its *residual (skip) connections* add the input of a
block to its output, which makes deep networks easier to train. Here the model is pretrained on ImageNet
(1000 classes). Reusing a pretrained network instead of training from scratch is called **transfer learning**.

The model is wrapped so that it accepts normal images with pixel values in `[0, 1]` and applies the ImageNet
normalisation internally. This keeps the attack strength `epsilon` in simple pixel units later on.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class NormalizedModel(nn.Module):
    """Wrap a model so it takes images in [0, 1] and normalises them internally."""

    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return self.model((x - self.mean) / self.std)


weights = models.ResNet18_Weights.IMAGENET1K_V1
base_resnet = models.resnet18(weights=weights)
model = NormalizedModel(base_resnet, IMAGENET_MEAN, IMAGENET_STD).to(DEVICE).eval()
CLASS_NAMES = weights.meta["categories"]

total_params = sum(p.numel() for p in base_resnet.parameters())
print(f"ResNet-18 loaded: {total_params / 1e6:.1f} million parameters, {len(CLASS_NAMES)} ImageNet classes")

In [ ]:
# How the image shrinks inside ResNet-18 (this decides the resolution of Grad-CAM later)
x = torch.zeros(1, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    x = base_resnet.maxpool(base_resnet.relu(base_resnet.bn1(base_resnet.conv1(x))))
    print("after stem  :", tuple(x.shape))
    for number, stage in enumerate([base_resnet.layer1, base_resnet.layer2,
                                    base_resnet.layer3, base_resnet.layer4], start=1):
        x = stage(x)
        print(f"after layer{number}: {tuple(x.shape)}")

**Load the images.** CIFAR-10 test images (32 x 32) are enlarged to 224 x 224, which is the input size ResNet expects.

In [ ]:
cifar_test = datasets.CIFAR10(root="data", train=False, download=True)
to_tensor_224 = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])


def load_cifar_batch(indices):
    """Return a batch tensor (N, 3, 224, 224) in [0, 1] and the CIFAR-10 class names."""
    images = torch.stack([to_tensor_224(cifar_test[i][0]) for i in indices])
    names = [cifar_test.classes[cifar_test[i][1]] for i in indices]
    return images, names


def to_numpy_image(tensor):
    """Convert a (3, H, W) tensor to an (H, W, 3) numpy array for plotting."""
    return tensor.detach().cpu().permute(1, 2, 0).numpy().clip(0, 1)


def predict(net, images, top_k=3):
    """Return top-k probabilities and class indices for a batch of images."""
    with torch.no_grad():
        probabilities = F.softmax(net(images.to(DEVICE)), dim=1)
    top_p, top_i = probabilities.topk(top_k, dim=1)
    return top_p.cpu(), top_i.cpu()


batch_images, batch_names = load_cifar_batch(range(8))
print("Batch shape:", tuple(batch_images.shape), "| CIFAR-10 labels:", batch_names)

In [ ]:
# Classify the 8 images and measure the inference time
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
top_p, top_i = predict(model, batch_images, top_k=3)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
inference_time = (time.perf_counter() - start) / len(batch_images)

rows = []
for n in range(len(batch_images)):
    rows.append({
        "CIFAR-10 label": batch_names[n],
        "Top-1 (ImageNet)": CLASS_NAMES[top_i[n, 0]], "Top-1 prob": round(top_p[n, 0].item(), 3),
        "Top-2": CLASS_NAMES[top_i[n, 1]], "Top-3": CLASS_NAMES[top_i[n, 2]],
    })
classification_table = pd.DataFrame(rows)
print(f"Inference time: {inference_time * 1000:.1f} ms per image on {DEVICE}")
classification_table

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for n, ax in enumerate(axes.ravel()):
    ax.imshow(to_numpy_image(batch_images[n]))
    ax.set_title(f"True: {batch_names[n]}\nPred: {CLASS_NAMES[top_i[n, 0]]} ({top_p[n, 0]:.2f})", fontsize=9)
    ax.axis("off")
plt.suptitle("ResNet-18 predictions (ImageNet classes) on CIFAR-10 images")
plt.tight_layout()
plt.savefig("outputs/resnet_classification.png", dpi=150, bbox_inches="tight")
plt.show()

**Note:** CIFAR-10 and ImageNet have different class lists (for example CIFAR-10 has "automobile", ImageNet has
"sports car", "convertible" and so on), so a prediction can be reasonable even if the words differ.
The 32 x 32 images are also very blurry after enlarging, which lowers confidence.

In [ ]:
# A sharper demo photo for the explainability experiments (falls back to a CIFAR-10 image if offline)
DEMO_URL = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"


def get_demo_image(path="data/dog.jpg"):
    """Download a sample photo; fall back to a CIFAR-10 test image if that fails."""
    try:
        if not os.path.exists(path):
            urllib.request.urlretrieve(DEMO_URL, path)
        return Image.open(path).convert("RGB"), "dog.jpg"
    except Exception as error:
        print("Download failed:", error, "-> using a CIFAR-10 test image instead")
        return cifar_test[0][0].convert("RGB"), "CIFAR-10 test image 0"


# To use your own photo in Colab, upload it and set demo_image = Image.open("your_file.jpg").convert("RGB")
demo_image, demo_name = get_demo_image()
demo_tensor = to_tensor_224(demo_image).unsqueeze(0)          # shape (1, 3, 224, 224)

demo_p, demo_i = predict(model, demo_tensor, top_k=5)
print(f"Demo image: {demo_name}")
for rank in range(5):
    print(f"  {rank + 1}. {CLASS_NAMES[demo_i[0, rank]]:<28} {demo_p[0, rank].item():.3f}")

## 2. CNN visualisation using Grad-CAM
**Grad-CAM** (Gradient-weighted Class Activation Mapping) shows *which regions of the image* the network used for
a chosen class.

1. Run the image through the network and pick a class score `y_c`.
2. Take the feature maps `A_k` of the last convolutional block.
3. Compute the gradient of `y_c` with respect to each feature map and average it spatially. This gives the
   importance weight `a_k` of every feature map.
4. Combine: `CAM = ReLU( sum_k a_k * A_k )`. The ReLU keeps only the regions that push the score up.
5. Enlarge the small map (7 x 7 for the last block) to the image size and overlay it as a heatmap.

In [ ]:
class GradCAM:
    """Grad-CAM: weight the feature maps of a layer by the average gradient of the class score."""

    def __init__(self, net, target_layer):
        self.net = net
        self.activations = None
        self.gradients = None
        self._handle = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output
        output.register_hook(self._save_gradient)   # called during backward()

    def _save_gradient(self, gradient):
        self.gradients = gradient

    def remove(self):
        """Remove the hook when it is no longer needed."""
        self._handle.remove()

    def __call__(self, image, class_idx=None):
        """Return (heatmap in [0, 1], class index, probability of that class)."""
        image = image.clone().detach().to(DEVICE).requires_grad_(True)
        self.net.zero_grad(set_to_none=True)
        logits = self.net(image)
        probabilities = F.softmax(logits, dim=1)
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        logits[0, class_idx].backward()

        feature_weights = self.gradients.mean(dim=(2, 3), keepdim=True)          # importance of each map
        cam = F.relu((feature_weights * self.activations.detach()).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=image.shape[2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), class_idx, float(probabilities[0, class_idx].item())


def overlay_heatmap(image_np, heatmap, alpha=0.5):
    """Blend a [0, 1] heatmap (jet colours) on top of an image."""
    colours = plt.cm.jet(heatmap)[..., :3]
    return np.clip((1 - alpha) * image_np + alpha * colours, 0, 1)


def run_gradcam(net, image, layer, class_idx=None):
    """Create a Grad-CAM object for `layer`, run it once and clean up."""
    explainer = GradCAM(net, layer)
    try:
        return explainer(image, class_idx)
    finally:
        explainer.remove()


LAST_LAYER = base_resnet.layer4[-1]     # last residual block (7 x 7 feature maps)
print("Grad-CAM ready. Target layer:", type(LAST_LAYER).__name__)

In [ ]:
# Explain the top-1 and the top-2 class of the demo image
top1, top2 = int(demo_i[0, 0]), int(demo_i[0, 1])
demo_np = to_numpy_image(demo_tensor[0])

cam_1, _, prob_1 = run_gradcam(model, demo_tensor, LAST_LAYER, top1)
cam_2, _, prob_2 = run_gradcam(model, demo_tensor, LAST_LAYER, top2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))
axes[0].imshow(demo_np)
axes[0].set_title("Input image")
axes[1].imshow(overlay_heatmap(demo_np, cam_1))
axes[1].set_title(f"Grad-CAM for '{CLASS_NAMES[top1]}' ({prob_1:.2f})")
axes[2].imshow(overlay_heatmap(demo_np, cam_2))
axes[2].set_title(f"Grad-CAM for '{CLASS_NAMES[top2]}' ({prob_2:.2f})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("outputs/gradcam_demo.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Grad-CAM on the 8 CIFAR-10 images
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for n in range(8):
    image_np = to_numpy_image(batch_images[n])
    cam, class_idx, prob = run_gradcam(model, batch_images[n:n + 1], LAST_LAYER)
    axes[0, n].imshow(image_np)
    axes[0, n].set_title(batch_names[n], fontsize=9)
    axes[1, n].imshow(overlay_heatmap(image_np, cam))
    axes[1, n].set_title(f"{CLASS_NAMES[class_idx][:16]} ({prob:.2f})", fontsize=8)
    axes[0, n].axis("off")
    axes[1, n].axis("off")
plt.suptitle("Grad-CAM on CIFAR-10 images (top row: input, bottom row: explanation)")
plt.tight_layout()
plt.savefig("outputs/gradcam_cifar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Which layer to explain? Compare an early, a middle and the last residual block
layers_to_compare = {
    "layer2 (28 x 28)": base_resnet.layer2[-1],
    "layer3 (14 x 14)": base_resnet.layer3[-1],
    "layer4 (7 x 7)": base_resnet.layer4[-1],
}
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
axes[0].imshow(demo_np)
axes[0].set_title("Input")
for ax, (name, layer) in zip(axes[1:], layers_to_compare.items()):
    cam, _, _ = run_gradcam(model, demo_tensor, layer, top1)
    ax.imshow(overlay_heatmap(demo_np, cam))
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.suptitle(f"Grad-CAM for '{CLASS_NAMES[top1]}' at different depths")
plt.tight_layout()
plt.savefig("outputs/gradcam_layers.png", dpi=150, bbox_inches="tight")
plt.show()

**How to read the result:** red/yellow regions are the parts of the image that increased the score of the chosen class.
Earlier layers have a finer map (28 x 28) but respond to simple patterns such as edges and textures. The last
block has a coarse map (7 x 7) but responds to object parts, which is why it is the usual choice for Grad-CAM.
Explaining the second-best class shows that the heatmap changes with the chosen class.

## 3. Adversarial attack using FGSM
The **Fast Gradient Sign Method** creates an adversarial image in one step:

`x_adv = clip( x + epsilon * sign( d loss / d x ), 0, 1 )`

Every pixel is moved by `epsilon` in the direction that *increases* the loss for the correct label. For a small
`epsilon` (such as 2/255 or 4/255) the change is almost invisible to humans, yet it can change the prediction.
Here the model's own clean prediction is used as the label of the demo image.

In [ ]:
def fgsm_attack(net, image, label, epsilon):
    """One-step FGSM attack. `image` is in [0, 1]; returns the adversarial image."""
    image = image.clone().detach().to(DEVICE).requires_grad_(True)
    loss = F.cross_entropy(net(image), label.to(DEVICE))
    net.zero_grad(set_to_none=True)
    loss.backward()
    adversarial = image + epsilon * image.grad.sign()
    return torch.clamp(adversarial, 0, 1).detach()


demo_label = torch.tensor([top1])
epsilons = [0, 1 / 255, 2 / 255, 4 / 255, 8 / 255, 16 / 255]

rows, adversarial_images = [], []
for eps in epsilons:
    adversarial = demo_tensor.to(DEVICE) if eps == 0 else fgsm_attack(model, demo_tensor, demo_label, eps)
    p, i = predict(model, adversarial, top_k=1)
    with torch.no_grad():
        original_class_prob = F.softmax(model(adversarial), dim=1)[0, top1].item()
    adversarial_images.append(adversarial.cpu())
    rows.append({"Epsilon": f"{round(eps * 255)}/255", "Predicted class": CLASS_NAMES[i[0, 0]],
                 "Confidence": round(p[0, 0].item(), 3),
                 "Prob. of original class": round(original_class_prob, 3),
                 "Prediction changed": bool(i[0, 0] != top1)})

fgsm_table = pd.DataFrame(rows)
fgsm_table

In [ ]:
fig, axes = plt.subplots(2, len(epsilons), figsize=(18, 6))
for col, eps in enumerate(epsilons):
    adv_np = to_numpy_image(adversarial_images[col][0])
    axes[0, col].imshow(adv_np)
    axes[0, col].set_title(f"eps = {round(eps * 255)}/255\n{fgsm_table.loc[col, 'Predicted class'][:18]}", fontsize=9)
    noise = adversarial_images[col][0] - demo_tensor[0]                     # the added perturbation
    noise_view = (noise / (2 * eps) + 0.5).clamp(0, 1) if eps > 0 else torch.full_like(noise, 0.5)
    axes[1, col].imshow(to_numpy_image(noise_view))
    axes[1, col].set_title("perturbation (amplified)", fontsize=8)
    axes[0, col].axis("off")
    axes[1, col].axis("off")
plt.suptitle("FGSM: adversarial images (top) and the noise that was added (bottom)")
plt.tight_layout()
plt.savefig("outputs/fgsm_examples.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6.5, 4))
plt.plot([round(e * 255) for e in epsilons], fgsm_table["Prob. of original class"], marker="o")
plt.xlabel("epsilon (out of 255)")
plt.ylabel("Probability of the original class")
plt.title("Confidence in the original class falls as epsilon grows")
plt.savefig("outputs/fgsm_confidence_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Does the explanation change after the attack? Compare Grad-CAM before and after.
changed = fgsm_table.index[fgsm_table["Prediction changed"]]
attack_position = int(changed[0]) if len(changed) > 0 else len(epsilons) - 1
adv_tensor = adversarial_images[attack_position]
adv_p, adv_i = predict(model, adv_tensor, top_k=1)
adv_class = int(adv_i[0, 0])

cam_before, _, _ = run_gradcam(model, demo_tensor, LAST_LAYER, top1)
cam_after, _, _ = run_gradcam(model, adv_tensor, LAST_LAYER, adv_class)

fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
axes[0].imshow(demo_np)
axes[0].set_title("Original image")
axes[1].imshow(overlay_heatmap(demo_np, cam_before))
axes[1].set_title(f"Explanation: '{CLASS_NAMES[top1][:20]}'")
axes[2].imshow(to_numpy_image(adv_tensor[0]))
axes[2].set_title(f"Adversarial (eps = {round(epsilons[attack_position] * 255)}/255)")
axes[3].imshow(overlay_heatmap(to_numpy_image(adv_tensor[0]), cam_after))
axes[3].set_title(f"Explanation: '{CLASS_NAMES[adv_class][:20]}'")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("outputs/gradcam_before_after_attack.png", dpi=150, bbox_inches="tight")
plt.show()

if len(changed) == 0:
    print("The prediction did not change even at the largest epsilon tested for this image.")
else:
    print(f"The prediction first changed at epsilon = {round(epsilons[attack_position] * 255)}/255: "
          f"'{CLASS_NAMES[top1]}' -> '{CLASS_NAMES[adv_class]}'")

## 4. Analysis: comparing Explainable AI techniques
Three techniques are compared on the same images:

| Technique | How it works | Needs gradients? |
|---|---|---|
| **Grad-CAM** | Weights the last feature maps by gradient averages | Yes |
| **Vanilla saliency** | Absolute gradient of the class score with respect to every input pixel | Yes |
| **Occlusion sensitivity** | Slides a grey patch over the image and records how much the class probability drops | No (only forward passes) |

**Quantitative test (deletion metric):** remove the pixels that a method marks as most important (replace them by
grey) and watch the probability of the predicted class. A good explanation makes the probability fall *quickly*.
A random heatmap is included as a reference. The score is the average probability over the deletion
fractions (**lower is better**).

In [ ]:
def saliency_map(net, image, class_idx):
    """Vanilla gradient saliency: |d score / d pixel|, maximum over the colour channels."""
    image = image.clone().detach().to(DEVICE).requires_grad_(True)
    score = net(image)[0, class_idx]
    net.zero_grad(set_to_none=True)
    score.backward()
    saliency = image.grad.abs().max(dim=1)[0][0].cpu().numpy()
    return (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)


def occlusion_map(net, image, class_idx, patch=32, stride=16, baseline=0.5):
    """Occlusion sensitivity: probability drop when a grey patch covers each region."""
    image = image.to(DEVICE)
    _, _, height, width = image.shape
    with torch.no_grad():
        base_prob = F.softmax(net(image), dim=1)[0, class_idx].item()

    positions = [(top, left) for top in range(0, height - patch + 1, stride)
                 for left in range(0, width - patch + 1, stride)]
    drops = []
    with torch.no_grad():
        for start in range(0, len(positions), 64):
            occluded = []
            for top, left in positions[start:start + 64]:
                copy = image.clone()
                copy[:, :, top:top + patch, left:left + patch] = baseline
                occluded.append(copy)
            probs = F.softmax(net(torch.cat(occluded, dim=0)), dim=1)[:, class_idx]
            drops.extend((base_prob - probs).cpu().tolist())

    heat = torch.zeros(height, width)
    counts = torch.zeros(height, width)
    for (top, left), drop in zip(positions, drops):
        heat[top:top + patch, left:left + patch] += drop
        counts[top:top + patch, left:left + patch] += 1
    heat = (heat / counts.clamp(min=1)).clamp(min=0).numpy()
    return (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)


def deletion_curve(net, image, class_idx, importance, fractions):
    """Class probability after removing the most important pixels first."""
    image = image.to(DEVICE)
    order = np.argsort(-importance.ravel())            # most important pixel first
    curve = []
    for fraction in fractions:
        removed = np.zeros(order.size, dtype=bool)
        removed[order[:int(fraction * order.size)]] = True
        mask = torch.from_numpy(removed.reshape(importance.shape)).to(DEVICE)
        masked = image.clone()
        masked[:, :, mask] = 0.5
        with torch.no_grad():
            curve.append(F.softmax(net(masked), dim=1)[0, class_idx].item())
    return curve


print("Explanation methods defined.")

In [ ]:
# Visual comparison on the demo image
rng = np.random.default_rng(SEED)
maps = {
    "Grad-CAM": cam_1,
    "Vanilla saliency": saliency_map(model, demo_tensor, top1),
    "Occlusion": occlusion_map(model, demo_tensor, top1),
}

fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
axes[0].imshow(demo_np)
axes[0].set_title("Input")
for ax, (name, heat) in zip(axes[1:], maps.items()):
    ax.imshow(overlay_heatmap(demo_np, heat))
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.suptitle(f"Three explanations for the class '{CLASS_NAMES[top1]}'")
plt.tight_layout()
plt.savefig("outputs/xai_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Quantitative comparison: deletion metric and run time over the 8 CIFAR-10 images
fractions = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7]
method_names = ["Grad-CAM", "Vanilla saliency", "Occlusion", "Random (reference)"]
curves = {name: [] for name in method_names}
times = {name: [] for name in method_names}

for n in range(len(batch_images)):
    image = batch_images[n:n + 1]
    class_idx = int(predict(model, image, top_k=1)[1][0, 0])

    def timed(function):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        result = function()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        return result, time.perf_counter() - start

    importance = {}
    importance["Grad-CAM"], times["Grad-CAM"] = (
        lambda r: (r[0][0], times["Grad-CAM"] + [r[1]]))(timed(lambda: run_gradcam(model, image, LAST_LAYER, class_idx)))
    importance["Vanilla saliency"], times["Vanilla saliency"] = (
        lambda r: (r[0], times["Vanilla saliency"] + [r[1]]))(timed(lambda: saliency_map(model, image, class_idx)))
    importance["Occlusion"], times["Occlusion"] = (
        lambda r: (r[0], times["Occlusion"] + [r[1]]))(timed(lambda: occlusion_map(model, image, class_idx)))
    importance["Random (reference)"] = rng.random((224, 224))

    for name in method_names:
        curves[name].append(deletion_curve(model, image, class_idx, importance[name], fractions))

summary_rows = []
for name in method_names:
    mean_curve = np.mean(curves[name], axis=0)
    summary_rows.append({
        "Method": name,
        "Deletion score (lower = better)": round(float(mean_curve.mean()), 4),
        "Time per image (s)": round(float(np.mean(times[name])), 4) if times[name] else 0.0,
    })
xai_summary = pd.DataFrame(summary_rows).sort_values("Deletion score (lower = better)").reset_index(drop=True)
xai_summary

In [ ]:
plt.figure(figsize=(7.5, 4.5))
for name in method_names:
    plt.plot([f * 100 for f in fractions], np.mean(curves[name], axis=0), marker="o", label=name)
plt.xlabel("Pixels removed (%), most important first")
plt.ylabel("Mean probability of the predicted class")
plt.title("Deletion test: a faster drop means a better explanation")
plt.legend()
plt.savefig("outputs/xai_deletion_curves.png", dpi=150, bbox_inches="tight")
plt.show()

best = xai_summary.iloc[0]
fastest = xai_summary[xai_summary["Method"] != "Random (reference)"].sort_values("Time per image (s)").iloc[0]
print(f"Best deletion score : {best['Method']} ({best['Deletion score (lower = better)']})")
print(f"Fastest method      : {fastest['Method']} ({fastest['Time per image (s)']} s per image)")

**Qualitative comparison (general properties of the three methods)**

| | Grad-CAM | Vanilla saliency | Occlusion |
|---|---|---|---|
| Resolution | Coarse (7 x 7 enlarged) | Pixel level | Depends on patch size (32 px here) |
| Class specific | Yes | Yes, but often noisy | Yes |
| Speed | One forward + one backward pass | One forward + one backward pass | Many forward passes (169 here) |
| Needs model internals | Needs a convolutional layer and gradients | Needs gradients | Works with any black-box model |
| Typical weakness | Blurry, tied to the chosen layer | Noisy, hard to read | Slow, depends on patch size and fill value |

The table above shows the general trade-offs. The deletion scores and times printed in the previous cells come from
this run, on only 8 images, so they can change with other images. Use them for your own observations.

## 5. Analysis: robustness against adversarial attacks
FGSM is a single step. **PGD** (Projected Gradient Descent) repeats small FGSM steps (here 10 steps of size
`epsilon / 4`) and projects the result back into the allowed `epsilon` range after every step. It is a stronger attack.

Because the CIFAR-10 images are not ImageNet classes, accuracy cannot be measured directly. Instead the
**prediction flip rate** is used: the fraction of images whose predicted class changes after the attack, compared
with the model's own clean prediction (higher flip rate = less robust).

In [ ]:
def pgd_attack(net, images, labels, epsilon, steps=10):
    """Iterative attack: small FGSM steps, each projected back to the epsilon ball."""
    step_size = epsilon / 4
    original = images.clone().detach().to(DEVICE)
    labels = labels.to(DEVICE)
    adversarial = original.clone()
    for _ in range(steps):
        adversarial.requires_grad_(True)
        loss = F.cross_entropy(net(adversarial), labels)
        net.zero_grad(set_to_none=True)
        loss.backward()
        adversarial = adversarial.detach() + step_size * adversarial.grad.sign()
        change = torch.clamp(adversarial - original, -epsilon, epsilon)
        adversarial = torch.clamp(original + change, 0, 1).detach()
    return adversarial


robust_images, _ = load_cifar_batch(range(16))
clean_labels = predict(model, robust_images, top_k=1)[1][:, 0]

rows = []
for eps in [0, 1 / 255, 2 / 255, 4 / 255, 8 / 255]:
    if eps == 0:
        fgsm_flip, pgd_flip = 0.0, 0.0
    else:
        fgsm_images = fgsm_attack(model, robust_images, clean_labels, eps)
        pgd_images = pgd_attack(model, robust_images, clean_labels, eps)
        fgsm_flip = (predict(model, fgsm_images, 1)[1][:, 0] != clean_labels).float().mean().item()
        pgd_flip = (predict(model, pgd_images, 1)[1][:, 0] != clean_labels).float().mean().item()
    rows.append({"Epsilon": f"{round(eps * 255)}/255", "FGSM flip rate": round(fgsm_flip, 3),
                 "PGD flip rate": round(pgd_flip, 3)})

robustness_table = pd.DataFrame(rows)
robustness_table

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(robustness_table["Epsilon"], robustness_table["FGSM flip rate"], marker="o", label="FGSM (1 step)")
plt.plot(robustness_table["Epsilon"], robustness_table["PGD flip rate"], marker="s", label="PGD (10 steps)")
plt.xlabel("epsilon")
plt.ylabel("Prediction flip rate (16 images)")
plt.title("Robustness of ResNet-18: FGSM vs PGD")
plt.legend()
plt.savefig("outputs/robustness_fgsm_pgd.png", dpi=150, bbox_inches="tight")
plt.show()

largest = robustness_table.iloc[-1]
print(f"At epsilon = {largest['Epsilon']}: FGSM changed {largest['FGSM flip rate'] * 100:.0f}% of the predictions, "
      f"PGD changed {largest['PGD flip rate'] * 100:.0f}%.")

**How to read the result:** the flip rate normally grows with `epsilon`, and PGD is normally at least as
effective as FGSM because it takes many small, corrected steps. A standard (not adversarially trained) network is
usually very fragile: perturbations of only a few intensity levels out of 255, which are hard to see, can change
many predictions. Defences include adversarial training (training on attacked images), input preprocessing and
detection of adversarial inputs.

## 6. Summary and learning outcomes
- Used a pretrained **ResNet-18** (transfer learning) and understood how the image shrinks through its stages.
- Implemented **Grad-CAM** from scratch with PyTorch hooks and compared the explanation at different depths.
- Implemented the **FGSM** attack, saw the prediction change for a nearly invisible perturbation and checked how
  the explanation changes after the attack.
- Compared **Grad-CAM, vanilla saliency and occlusion** both visually and with a quantitative deletion test.
- Compared **FGSM and PGD** using the prediction flip rate to analyse robustness.
- Learned that high accuracy does not mean robustness, and that explanations help to understand and debug a model.